# Peramalan Tingkat Produksi Padi dengan Singular Spectrum Analysis (SSA)

**Lokasi penelitian:** Pabrik KP. Putra Yusima, Desa Masjid, Kecamatan Meurah Mulia, Kabupaten Aceh Utara
**Data:** produksi padi harian (ton) tahun 2024

Notebook ini menjalankan setiap tahap SSA secara berurutan dan menampilkan hasil antaranya, sesuai langkah penelitian pada Gambar 3.1:

1. **Pengumpulan data**: membaca data mentah, membersihkan *missing values*, dan menguji stasioneritas
2. **Implementasi SSA**
   - Dekomposisi: *embedding* dan *Singular Value Decomposition* (SVD)
   - Rekonstruksi: *grouping* dan *diagonal averaging*
3. **Peramalan**: Recurrent SSA (R-SSA)
4. **Evaluasi**: MAE, MSE, RMSE, dan MAPE dengan kriteria Tabel 2.1

Semua perhitungan memakai modul `ssa.py` yang juga dipakai aplikasi web. Karena itu, angka di notebook ini sama dengan hasil menu **Proses Peramalan** pada mode otomatis.
Semua grafik interaktif: arahkan kursor untuk melihat nilai, seret untuk memperbesar, klik dua kali untuk kembali, dan klik legenda untuk menyembunyikan atau menampilkan seri.

In [1]:
import numpy as np
import pandas as pd
import plotly.graph_objects as go
import plotly.express as px
import plotly.io as pio
from plotly.subplots import make_subplots
from statsmodels.tsa.stattools import adfuller

import ssa
from data_import import read_file

pio.renderers.default = 'plotly_mimetype+notebook_connected'
pio.templates.default = 'plotly_white'
pd.set_option('display.float_format', lambda v: f'{v:,.4f}')
pd.set_option('display.max_columns', 30)

RASIO_LATIH = 0.8   # pembagian data latih : uji = 80 : 20 (bawaan aplikasi)
HORIZON = 365       # ramalan 12 bulan ke depan (Bab 3, Tahap Ramalan)
R_MAKS = 10         # batas atas r saat pencarian parameter (sama dengan aplikasi)

NAMA_BULAN = ['Januari', 'Februari', 'Maret', 'April', 'Mei', 'Juni', 'Juli',
              'Agustus', 'September', 'Oktober', 'November', 'Desember']

print(f'numpy {np.__version__} | pandas {pd.__version__}')
print(f'Rasio latih {RASIO_LATIH:.0%}, horizon ramalan {HORIZON} hari, r maksimum {R_MAKS}')

numpy 2.4.6 | pandas 3.0.6
Rasio latih 80%, horizon ramalan 365 hari, r maksimum 10


## 1. Pengumpulan Data

### 1.1 Data mentah dari pabrik
Data diterima dalam bentuk Excel dengan beberapa blok tabel yang disusun berdampingan, masing-masing berkolom `No | Bulan | Ton | Produksi Hari Sebelumnya | Rata-rata 7 Hari`.
Yang dipakai hanya kolom **Ton**, yaitu produksi pada hari tersebut. Dua kolom lainnya diturunkan dari kolom Ton.

In [2]:
raw = pd.read_excel('data/raw/data_mentahan_2024.xlsx', header=None)
print('Ukuran sheet mentah (baris x kolom):', raw.shape)
raw.iloc[:12, :12].fillna('')

Ukuran sheet mentah (baris x kolom): (133, 19)


,0,1,2,3,4,5,6,7,8,9,10,11
0,,,,,,,,,Data Mentahan Produksi Padi Tahun 2024,,,
1,,,,,,,,,,,,
2,,,,,,,,,,,,
3,No,Bulan,Ton,Produksi Hari Sebelumnya,Rata-rata 7 Hari,,,No,Bulan,Ton,Produksi Hari Sebelumnya,Rata-rata 7 Hari
4,1,Januari,58,-,-,,,1,Mei,62,66,65.2900
5,2,Januari,58,58,58,,,2,Mei,63,62,65
6,3,Januari,57,58,58,,,3,Mei,64,63,64.5700
7,4,Januari,59,57,57.6700,,,4,Mei,60,64,64.1400
8,5,Januari,54,59,58,,,5,Mei,62,60,63.1400
9,6,Januari,60,54,57.2000,,,6,Mei,61,62,62.5700


### 1.2 Konversi menjadi deret waktu harian
Setiap blok dibaca, nomor hari digabung dengan nama bulan dan tahun dari judul ("Tahun 2024"), lalu diurutkan menurut tanggal.
Hasilnya adalah deret $X = (x_1, x_2, \dots, x_N)$.

In [3]:
df = read_file('data/raw/data_mentahan_2024.xlsx', 'data_mentahan_2024.xlsx')
df.insert(0, 't', np.arange(1, len(df) + 1))
x = df['ton'].to_numpy()
N = len(x)

csv = pd.read_csv('data/produksi_padi_2024.csv', parse_dates=['tanggal'])
print(f'N = {N} hari ({df.tanggal.min():%d-%m-%Y} s.d. {df.tanggal.max():%d-%m-%Y})')
print('Sama dengan data/produksi_padi_2024.csv (data awal aplikasi):',
      csv['tanggal'].equals(df['tanggal']) and np.allclose(csv['ton'], x))

display(df.head(10))
display(df.tail(5))

N = 366 hari (01-01-2024 s.d. 31-12-2024)
Sama dengan data/produksi_padi_2024.csv (data awal aplikasi): True


,t,tanggal,ton
0,1,2024-01-01,58.0000
1,2,2024-01-02,58.0000
2,3,2024-01-03,57.0000
3,4,2024-01-04,59.0000
4,5,2024-01-05,54.0000
5,6,2024-01-06,60.0000
6,7,2024-01-07,57.0000
7,8,2024-01-08,59.0000
8,9,2024-01-09,57.0000
9,10,2024-01-10,56.0000


,t,tanggal,ton
361,362,2024-12-27,63.0000
362,363,2024-12-28,56.0000
363,364,2024-12-29,61.0000
364,365,2024-12-30,56.0000
365,366,2024-12-31,57.0000


### 1.3 Pembersihan data
SSA membutuhkan deret waktu dengan panjang $N$ **tanpa data hilang** (Bab 2.4). Pemeriksaan di bawah memastikan tidak ada nilai kosong, tanggal ganda, maupun hari yang terlewat.

In [4]:
rentang = pd.date_range(df.tanggal.min(), df.tanggal.max(), freq='D')
tanggal_hilang = rentang.difference(df.tanggal)

cek = pd.DataFrame({
    'Pemeriksaan': ['Jumlah data (N)', 'Jumlah hari dalam rentang tanggal', 'Nilai kosong (missing value)',
                    'Tanggal ganda', 'Tanggal yang hilang', 'Produksi negatif', 'Produksi nol'],
    'Hasil': [N, len(rentang), int(df['ton'].isna().sum()), int(df['tanggal'].duplicated().sum()),
              len(tanggal_hilang), int((df['ton'] < 0).sum()), int((df['ton'] == 0).sum())],
})
display(cek)
print('Data siap diproses:' , cek['Hasil'].iloc[2:].sum() == 0 and N == len(rentang))

,Pemeriksaan,Hasil
0,Jumlah data (N),366
1,Jumlah hari dalam rentang tanggal,366
2,Nilai kosong (missing value),0
3,Tanggal ganda,0
4,Tanggal yang hilang,0
5,Produksi negatif,0
6,Produksi nol,0


Data siap diproses: True


In [5]:
statistik = df['ton'].describe().rename({
    'count': 'Jumlah data', 'mean': 'Rata-rata', 'std': 'Standar deviasi', 'min': 'Minimum',
    '25%': 'Kuartil 1', '50%': 'Median', '75%': 'Kuartil 3', 'max': 'Maksimum'})
statistik['Total produksi'] = df['ton'].sum()
statistik.to_frame('Produksi (ton)')

,Produksi (ton)
Jumlah data,366.0000
Rata-rata,61.0656
Standar deviasi,3.7891
Minimum,53.0000
Kuartil 1,58.0000
Median,61.0000
Kuartil 3,64.0000
Maksimum,69.0000
Total produksi,"22,350.0000"


In [6]:
fig = go.Figure(go.Scatter(x=df.tanggal, y=df.ton, mode='lines+markers', marker_size=3,
                           name='Produksi', hovertemplate='%{x|%d %b %Y}<br>%{y:.0f} ton<extra></extra>'))
fig.update_layout(title='Produksi Padi Harian Tahun 2024', xaxis_title='Tanggal', yaxis_title='Produksi (ton)',
                  xaxis_rangeslider_visible=True, height=480)
fig.show()

In [7]:
bulanan = (df.groupby(df.tanggal.dt.month)['ton']
             .agg(hari='count', total='sum', rata_rata='mean', minimum='min', maksimum='max'))
bulanan.index = [NAMA_BULAN[m - 1] for m in bulanan.index]
bulanan.index.name = 'Bulan'
display(bulanan)

fig = px.bar(bulanan.reset_index(), x='Bulan', y='total', text_auto='.0f',
             labels={'total': 'Total produksi (ton)'}, title='Total Produksi per Bulan (2024)')
fig.update_traces(hovertemplate='%{x}<br>%{y:,.0f} ton<extra></extra>')
fig.show()

,hari,total,rata_rata,minimum,maksimum
Bulan,,,,,
Januari,31,"1,778.0000",57.3548,54.0000,61.0000
Februari,29,"1,663.0000",57.3448,53.0000,61.0000
Maret,31,"1,990.0000",64.1935,61.0000,69.0000
April,30,"1,957.0000",65.2333,61.0000,69.0000
Mei,31,"1,865.0000",60.1613,56.0000,64.0000
Juni,30,"1,796.0000",59.8667,56.0000,64.0000
Juli,31,"1,856.0000",59.8710,56.0000,64.0000
Agustus,31,"1,844.0000",59.4839,56.0000,64.0000
September,30,"1,964.0000",65.4667,61.0000,69.0000


### 1.4 Pengecekan stasioneritas
SSA adalah metode nonparametrik sehingga **tidak mensyaratkan data stasioner**. Pengecekan ini hanya untuk menggambarkan karakter data (Bab 3, langkah 1c).
Uji yang dipakai adalah *Augmented Dickey-Fuller* (ADF) dengan hipotesis:
- $H_0$: data tidak stasioner (memiliki *unit root*)
- $H_1$: data stasioner

$H_0$ ditolak jika *p-value* < 0,05.

In [8]:
adf_stat, p_value, lag, n_obs, kritis, _ = adfuller(x, autolag='AIC', result_object=False)
display(pd.DataFrame({
    'Nilai': [adf_stat, p_value, lag, n_obs, kritis['1%'], kritis['5%'], kritis['10%']],
}, index=['Statistik ADF', 'p-value', 'Lag yang dipakai', 'Jumlah observasi',
          'Nilai kritis 1%', 'Nilai kritis 5%', 'Nilai kritis 10%']))
print('Kesimpulan:', 'tolak H0 -> data stasioner' if p_value < 0.05 else 'gagal tolak H0 -> data tidak stasioner',
      f'(p-value = {p_value:.4f})')

,Nilai
Statistik ADF,-3.4750
p-value,0.0087
Lag yang dipakai,4.0000
Jumlah observasi,361.0000
Nilai kritis 1%,-3.4486
Nilai kritis 5%,-2.8696
Nilai kritis 10%,-2.5711


Kesimpulan: tolak H0 -> data stasioner (p-value = 0.0087)


## 2. Pembagian Data Latih dan Data Uji
Data dibagi **berurutan menurut waktu** (tidak diacak) karena peramalan selalu memakai data masa lalu untuk menebak masa depan.
Model dibangun dari data latih, lalu dipakai untuk meramal periode data uji. Hasil ramalan itu dibandingkan dengan nilai aktual untuk menghitung akurasi.

In [9]:
n_latih = int(round(N * RASIO_LATIH))
n_uji = N - n_latih
latih, uji = x[:n_latih], x[n_latih:]
tgl_latih, tgl_uji = df.tanggal.iloc[:n_latih], df.tanggal.iloc[n_latih:]

display(pd.DataFrame({
    'Jumlah hari': [n_latih, n_uji, N],
    'Tanggal awal': [tgl_latih.iloc[0].date(), tgl_uji.iloc[0].date(), df.tanggal.iloc[0].date()],
    'Tanggal akhir': [tgl_latih.iloc[-1].date(), tgl_uji.iloc[-1].date(), df.tanggal.iloc[-1].date()],
    'Persentase': [f'{n_latih / N:.1%}', f'{n_uji / N:.1%}', '100%'],
}, index=['Data latih', 'Data uji', 'Total']))

fig = go.Figure()
fig.add_scatter(x=tgl_latih, y=latih, name='Data latih', mode='lines')
fig.add_scatter(x=tgl_uji, y=uji, name='Data uji', mode='lines')
fig.update_layout(title='Pembagian Data Latih dan Data Uji', xaxis_title='Tanggal', yaxis_title='Produksi (ton)',
                  hovermode='x unified', height=420)
fig.show()

,Jumlah hari,Tanggal awal,Tanggal akhir,Persentase
Data latih,293,2024-01-01,2024-10-19,80.1%
Data uji,73,2024-10-20,2024-12-31,19.9%
Total,366,2024-01-01,2024-12-31,100%


## 3. Penentuan Window Length ($L$) dan Grouping Effect ($r$)
Belum ada metode baku untuk menentukan $L$, sehingga $L$ ditentukan dengan **coba-coba** (Bab 2.4).
Setiap $L$ dari 2 sampai $N_{latih}/2$ dicoba dengan setiap $r$ dari 1 sampai $\min(L-1, 10)$. Untuk setiap kombinasi, SSA dilatih pada data latih, lalu dipakai meramal data uji dan dihitung MAPE-nya.
Kombinasi dengan **MAPE terkecil** dipilih. Cara ini sama persis dengan mode otomatis pada aplikasi.

In [10]:
baris = []
for L_coba in range(2, n_latih // 2 + 1):
    for r_coba in range(1, min(L_coba - 1, R_MAKS) + 1):
        try:
            ev = ssa.evaluate(x, L_coba, r_coba, RASIO_LATIH)
        except (ValueError, np.linalg.LinAlgError):
            continue  # kombinasi tidak dapat diramal (verticality coefficient >= 1)
        if np.isfinite(ev['mape']):
            baris.append({'L': L_coba, 'r': r_coba, 'MAE': ev['mae'], 'RMSE': ev['rmse'], 'MAPE (%)': ev['mape']})
grid = pd.DataFrame(baris)

L, r, _ = ssa.search_parameters(x, RASIO_LATIH, r_max=R_MAKS)
terbaik = grid.loc[grid['MAPE (%)'].idxmin()]
assert (terbaik['L'], terbaik['r']) == (L, r)

print(f'Kombinasi yang diuji : {len(grid)}')
print(f'Rentang L            : 2 s.d. {n_latih // 2}')
print(f'Parameter terpilih   : L = {L}, r = {r}, MAPE data uji = {terbaik["MAPE (%)"]:.4f}%')

Kombinasi yang diuji : 1405
Rentang L            : 2 s.d. 146
Parameter terpilih   : L = 16, r = 2, MAPE data uji = 3.7877%


In [11]:
per_L = grid.loc[grid.groupby('L')['MAPE (%)'].idxmin()].reset_index(drop=True)
print('10 window length dengan MAPE terkecil (r terbaik untuk masing-masing L):')
per_L.sort_values('MAPE (%)').head(10).reset_index(drop=True)

10 window length dengan MAPE terkecil (r terbaik untuk masing-masing L):


,L,r,MAE,RMSE,MAPE (%)
0,16,2,2.3062,2.8662,3.7877
1,17,3,2.3031,2.7828,3.7953
2,36,10,2.3022,3.0122,3.8161
3,37,10,2.3285,3.0077,3.8719
4,44,4,2.3336,2.8655,3.9085
5,18,2,2.4363,2.8864,4.1062
6,15,2,2.4731,2.9226,4.1682
7,7,2,2.4868,2.9543,4.1929
8,35,8,2.4822,3.0403,4.2062
9,125,6,2.5209,2.9496,4.2129


In [12]:
fig = go.Figure()
fig.add_scatter(x=per_L['L'], y=per_L['MAPE (%)'], mode='lines+markers', name='MAPE terbaik',
                customdata=per_L[['r', 'MAE']],
                hovertemplate='L = %{x}<br>r = %{customdata[0]}<br>MAPE = %{y:.4f}%<br>MAE = %{customdata[1]:.4f}<extra></extra>')
fig.add_scatter(x=[L], y=[terbaik['MAPE (%)']], mode='markers', name=f'Terpilih (L={L}, r={r})',
                marker=dict(size=14, symbol='star', color='crimson'))
fig.update_layout(title='MAPE Data Uji untuk Setiap Window Length (L)', xaxis_title='Window length (L)',
                  yaxis_title='MAPE (%)', height=450)
fig.show()

In [13]:
peta = grid.pivot(index='r', columns='L', values='MAPE (%)')
batas = np.nanpercentile(peta.to_numpy(), 90)  # warna dipotong agar kombinasi ekstrem tidak menenggelamkan pola
fig = px.imshow(peta, aspect='auto', origin='lower', color_continuous_scale='Viridis_r', zmax=batas,
                labels=dict(x='Window length (L)', y='Grouping effect (r)', color='MAPE (%)'),
                title='Peta MAPE Data Uji untuk Setiap Kombinasi L dan r')
fig.update_traces(hovertemplate='L = %{x}<br>r = %{y}<br>MAPE = %{z:.4f}%<extra></extra>')
fig.add_scatter(x=[L], y=[r], mode='markers', marker=dict(size=14, symbol='star', color='crimson'),
                name='Terpilih', hovertemplate=f'Terpilih: L = {L}, r = {r}<extra></extra>')
fig.update_layout(height=450)
fig.show()

## 4. Tahap Dekomposisi

### 4.1 Embedding
Data latih sepanjang $N$ diubah menjadi **matriks lintasan** (*trajectory matrix*) berukuran $L \times K$ dengan $K = N - L + 1$ (persamaan 2.5):

$$
\mathbf{X} = [X_1, X_2, \dots, X_K] =
\begin{bmatrix}
x_1 & x_2 & \cdots & x_K \\
x_2 & x_3 & \cdots & x_{K+1} \\
\vdots & \vdots & \ddots & \vdots \\
x_L & x_{L+1} & \cdots & x_N
\end{bmatrix}
$$

Matriks ini adalah **matriks Hankel**: semua elemen pada satu anti-diagonal bernilai sama.

In [14]:
X = ssa.embedding(latih, L)
K = X.shape[1]
print(f'N (data latih) = {n_latih}')
print(f'L = {L}')
print(f'K = N - L + 1 = {n_latih} - {L} + 1 = {K}')
print(f'Ukuran matriks lintasan: {X.shape[0]} x {X.shape[1]}')
print('Matriks Hankel (X[i, j] = x_(i+j+1)):',
      all(X[i, j] == latih[i + j] for i in range(L) for j in range(K)))

matriks = pd.DataFrame(X, index=[f'baris {i + 1}' for i in range(L)], columns=[f'X{j + 1}' for j in range(K)])
print(f'\n{L} baris pertama x 10 kolom pertama:')
matriks.iloc[:, :10].style.format('{:.0f}')

N (data latih) = 293
L = 16
K = N - L + 1 = 293 - 16 + 1 = 278
Ukuran matriks lintasan: 16 x 278
Matriks Hankel (X[i, j] = x_(i+j+1)): True

16 baris pertama x 10 kolom pertama:


,X1,X2,X3,X4,X5,X6,X7,X8,X9,X10
baris 1,58,58,57,59,54,60,57,59,57,56
baris 2,58,57,59,54,60,57,59,57,56,61
baris 3,57,59,54,60,57,59,57,56,61,55
baris 4,59,54,60,57,59,57,56,61,55,56
baris 5,54,60,57,59,57,56,61,55,56,55
baris 6,60,57,59,57,56,61,55,56,55,59
baris 7,57,59,57,56,61,55,56,55,59,58
baris 8,59,57,56,61,55,56,55,59,58,58
baris 9,57,56,61,55,56,55,59,58,58,61
baris 10,56,61,55,56,55,59,58,58,61,55


In [15]:
fig = px.imshow(X, aspect='auto', color_continuous_scale='YlGn',
                labels=dict(x='Kolom (j = 1..K)', y='Baris (i = 1..L)', color='Ton'),
                x=np.arange(1, K + 1), y=np.arange(1, L + 1),
                title=f'Matriks Lintasan {L} x {K} (pola garis miring = sifat Hankel)')
fig.update_traces(hovertemplate='baris %{y}, kolom %{x}<br>%{z:.0f} ton<extra></extra>')
fig.update_layout(height=420)
fig.show()

### 4.2 Singular Value Decomposition (SVD)
Matriks lintasan diuraikan menjadi

$$\mathbf{X} = \sum_{i=1}^{d} \sqrt{\lambda_i}\, U_i V_i^{T} = \mathbf{X}_1 + \mathbf{X}_2 + \dots + \mathbf{X}_d$$

dengan $\sqrt{\lambda_i}$ adalah **nilai singular** (diurutkan dari terbesar), $U_i$ vektor singular kiri (vektor eigen dari $\mathbf{X}\mathbf{X}^T$), dan $V_i$ vektor singular kanan.
Setiap $(\sqrt{\lambda_i}, U_i, V_i)$ disebut **eigentriple**. Kontribusi komponen ke-$i$ terhadap data adalah $\lambda_i / \sum \lambda_j$.

In [16]:
U, s, Vt = ssa.svd(X)
eigen = s ** 2
svd_tabel = pd.DataFrame({
    'Nilai singular (√λ)': s,
    'Nilai eigen (λ)': eigen,
    'Kontribusi (%)': eigen / eigen.sum() * 100,
    'Kumulatif (%)': np.cumsum(eigen) / eigen.sum() * 100,
}, index=pd.Index(np.arange(1, len(s) + 1), name='Komponen'))

print(f'Jumlah eigentriple d = {len(s)}')
print('X = U · diag(s) · Vᵀ terpenuhi:', np.allclose(X, U @ np.diag(s) @ Vt))
svd_tabel

Jumlah eigentriple d = 16
X = U · diag(s) · Vᵀ terpenuhi: True


,Nilai singular (√λ),Nilai eigen (λ),Kontribusi (%),Kumulatif (%)
Komponen,,,,
1,"4,091.0262","16,736,495.4185",99.8153,99.8153
2,69.9907,"4,898.6989",0.0292,99.8446
3,51.9504,"2,698.8426",0.0161,99.8607
4,50.5516,"2,555.4620",0.0152,99.8759
5,44.8681,"2,013.1506",0.0120,99.8879
6,44.0802,"1,943.0649",0.0116,99.8995
7,44.0049,"1,936.4311",0.0115,99.9110
8,42.8452,"1,835.7093",0.0109,99.9220
9,42.7177,"1,824.8061",0.0109,99.9329


In [17]:
fig = make_subplots(rows=1, cols=2, subplot_titles=('Nilai Singular (skala log)', 'Kontribusi Kumulatif'))
fig.add_scatter(x=svd_tabel.index, y=s, mode='lines+markers', name='Nilai singular',
                hovertemplate='Komponen %{x}<br>√λ = %{y:.4f}<extra></extra>', row=1, col=1)
fig.add_bar(x=svd_tabel.index, y=svd_tabel['Kontribusi (%)'], name='Kontribusi (%)',
            hovertemplate='Komponen %{x}<br>%{y:.4f}%<extra></extra>', row=1, col=2)
fig.add_scatter(x=svd_tabel.index, y=svd_tabel['Kumulatif (%)'], mode='lines+markers', name='Kumulatif (%)',
                hovertemplate='Komponen 1..%{x}<br>%{y:.4f}%<extra></extra>', row=1, col=2)
fig.update_yaxes(type='log', title_text='√λ', row=1, col=1)
fig.update_yaxes(title_text='%', row=1, col=2)
fig.update_xaxes(title_text='Komponen ke-i', dtick=1)
fig.update_layout(height=420, title='Hasil SVD Matriks Lintasan')
fig.show()

In [18]:
fig = go.Figure()
for i in range(min(6, L)):
    fig.add_scatter(x=np.arange(1, L + 1), y=U[:, i], mode='lines+markers', name=f'U{i + 1}',
                    visible=True if i < 3 else 'legendonly',
                    hovertemplate=f'U{i + 1}' + '<br>elemen %{x}: %{y:.4f}<extra></extra>')
fig.update_layout(title='Vektor Singular Kiri (Vektor Eigen) U₁–U₆, klik legenda untuk menampilkan',
                  xaxis_title='Elemen ke- (1..L)', yaxis_title='Nilai', height=420)
fig.show()

## 5. Tahap Rekonstruksi

### 5.1 Grouping
Setiap matriks elementer $\mathbf{X}_i = \sqrt{\lambda_i} U_i V_i^T$ diubah menjadi deret elementer $\tilde{x}^{(i)}$ (lewat *diagonal averaging*, lihat 5.2). Deret-deret ini lalu dikelompokkan:

| Kelompok | Komponen | Peran |
|---|---|---|
| Tren | 1 | Pola jangka panjang (tingkat produksi) |
| Musiman / osilasi | 2 … $r$ | Fluktuasi berulang |
| Noise | $r+1$ … $d$ | Dibuang |

Nilai $r$ diambil dari hasil coba-coba di Bagian 3. Untuk melihat komponen mana yang saling terkait, dipakai **matriks w-correlation**. Nilai yang mendekati 1 berarti dua komponen membentuk satu pola yang sama, sedangkan nilai yang mendekati 0 berarti keduanya terpisah dengan baik.

In [19]:
komponen = ssa.elementary_components(U, s, Vt, len(s))

def w_correlation(comps, L, K):
    n = len(comps[0])
    w = np.array([min(i + 1, L, K, n - i) for i in range(n)])  # bobot = banyaknya elemen per anti-diagonal
    F = np.array(comps)
    norma = np.sqrt((w * F ** 2).sum(axis=1))
    return np.abs((w * F) @ F.T / np.outer(norma, norma))

wcorr = w_correlation(komponen, L, K)
label = [f'F{i + 1}' for i in range(len(s))]
fig = px.imshow(wcorr, x=label, y=label, zmin=0, zmax=1, color_continuous_scale='Greys', text_auto='.2f',
                title='Matriks w-Correlation Komponen Elementer')
fig.update_traces(hovertemplate='%{y} vs %{x}<br>w-corr = %{z:.4f}<extra></extra>')
fig.update_layout(height=600)
fig.show()

In [20]:
n_tampil = min(6, len(komponen))
fig = make_subplots(rows=n_tampil, cols=1, shared_xaxes=True, vertical_spacing=0.03,
                    subplot_titles=[f'Komponen {i + 1} (kontribusi {svd_tabel["Kontribusi (%)"].iloc[i]:.4f}%)'
                                    for i in range(n_tampil)])
for i in range(n_tampil):
    peran = 'tren' if i == 0 else ('musiman' if i < r else 'noise')
    fig.add_scatter(x=tgl_latih, y=komponen[i], mode='lines', name=f'F{i + 1} ({peran})',
                    hovertemplate='%{x|%d %b %Y}<br>%{y:.4f}<extra></extra>', row=i + 1, col=1)
fig.update_layout(height=170 * n_tampil, title='Deret Komponen Elementer pada Data Latih')
fig.show()

In [21]:
grouping_tabel = pd.DataFrame({
    'Kelompok': ['Tren', 'Musiman', 'Noise (dibuang)'],
    'Komponen': ['1', f'2..{r}' if r > 2 else ('2' if r == 2 else '-'), f'{r + 1}..{len(s)}'],
    'Kontribusi (%)': [svd_tabel['Kontribusi (%)'].iloc[0],
                       svd_tabel['Kontribusi (%)'].iloc[1:r].sum(),
                       svd_tabel['Kontribusi (%)'].iloc[r:].sum()],
})
print(f'Grouping effect r = {r}')
grouping_tabel

Grouping effect r = 2


,Kelompok,Komponen,Kontribusi (%)
0,Tren,1,99.8153
1,Musiman,2,0.0292
2,Noise (dibuang),3..16,0.1554


### 5.2 Diagonal Averaging
Matriks hasil *grouping* dikembalikan menjadi deret sepanjang $N$. Caranya, setiap anti-diagonal ($i + j$ konstan) dirata-ratakan:

$$\tilde{x}_k = \frac{1}{|A_k|} \sum_{(i,j) \in A_k} y_{ij}, \qquad A_k = \{(i,j) : i + j = k + 1\}$$

Contoh di bawah menghitung tiga nilai pertama deret komponen 1 secara manual dari matriks $\mathbf{X}_1 = \sqrt{\lambda_1} U_1 V_1^T$.

In [22]:
X1 = s[0] * np.outer(U[:, 0], Vt[0])
print('Potongan matriks elementer X1 (5 x 5 pertama):')
display(pd.DataFrame(X1[:5, :5], index=[f'i={i + 1}' for i in range(5)], columns=[f'j={j + 1}' for j in range(5)]))

for k in range(3):
    anggota = [X1[i, k - i] for i in range(k + 1)]
    rumus = ' + '.join(f'{v:.4f}' for v in anggota)
    print(f'x̃_{k + 1} = ({rumus}) / {len(anggota)} = {np.mean(anggota):.4f}')
print('Hasil fungsi diagonal_averaging     :', np.round(ssa.diagonal_averaging(X1)[:3], 4))

Potongan matriks elementer X1 (5 x 5 pertama):


,j=1,j=2,j=3,j=4,j=5
i=1,57.2381,57.2382,57.4259,57.3008,57.1763
i=2,57.2654,57.2655,57.4533,57.3281,57.2036
i=3,57.2958,57.2959,57.4838,57.3585,57.2340
i=4,57.3244,57.3245,57.5125,57.3872,57.2625
i=5,57.3470,57.3471,57.5351,57.4097,57.2851


x̃_1 = (57.2381) / 1 = 57.2381
x̃_2 = (57.2382 + 57.2654) / 2 = 57.2518
x̃_3 = (57.4259 + 57.2655 + 57.2958) / 3 = 57.3291
Hasil fungsi diagonal_averaging     : [57.2381 57.2518 57.3291]


In [23]:
model = ssa.fit(latih, L, r)
rekon = pd.DataFrame({
    'tanggal': tgl_latih.dt.date.to_numpy(),
    'aktual': latih,
    'tren': model['trend'],
    'musiman': model['seasonal'],
    'rekonstruksi': model['signal'],
    'noise': model['noise'],
})
print('rekonstruksi = tren + musiman      :', np.allclose(rekon.rekonstruksi, rekon.tren + rekon.musiman))
print('aktual = rekonstruksi + noise      :', np.allclose(rekon.aktual, rekon.rekonstruksi + rekon.noise))
print('aktual = jumlah seluruh komponen   :', np.allclose(latih, np.sum(komponen, axis=0)))
print(f'\n15 baris pertama hasil grouping & diagonal averaging (dari {len(rekon)} baris):')
rekon.head(15)

rekonstruksi = tren + musiman      : True
aktual = rekonstruksi + noise      : True
aktual = jumlah seluruh komponen   : True

15 baris pertama hasil grouping & diagonal averaging (dari 293 baris):


,tanggal,aktual,tren,musiman,rekonstruksi,noise
0,2024-01-01,58.0000,57.2381,0.4911,57.7292,0.2708
1,2024-01-02,58.0000,57.2518,0.3736,57.6254,0.3746
2,2024-01-03,57.0000,57.3291,0.1280,57.4571,-0.4571
3,2024-01-04,59.0000,57.3436,0.1114,57.4550,1.5450
4,2024-01-05,54.0000,57.3319,0.0720,57.4039,-3.4039
5,2024-01-06,60.0000,57.3631,0.1352,57.4983,2.5017
6,2024-01-07,57.0000,57.3424,0.1719,57.5143,-0.5143
7,2024-01-08,59.0000,57.3320,0.1992,57.5312,1.4688
8,2024-01-09,57.0000,57.3116,0.1892,57.5008,-0.5008
9,2024-01-10,56.0000,57.2847,0.2096,57.4943,-1.4943


In [24]:
fig = make_subplots(rows=2, cols=1, shared_xaxes=True, row_heights=[0.7, 0.3], vertical_spacing=0.06,
                    subplot_titles=('Data Aktual vs Hasil Rekonstruksi', 'Noise (komponen yang dibuang)'))
fig.add_scatter(x=tgl_latih, y=latih, name='Aktual', mode='lines', line=dict(color='lightgray'), row=1, col=1)
fig.add_scatter(x=tgl_latih, y=model['signal'], name=f'Rekonstruksi (r={r})', mode='lines', row=1, col=1)
fig.add_scatter(x=tgl_latih, y=model['trend'], name='Tren (komponen 1)', mode='lines',
                line=dict(dash='dash'), row=1, col=1)
fig.add_scatter(x=tgl_latih, y=model['noise'], name='Noise', mode='lines', line=dict(color='gray'), row=2, col=1)
fig.update_traces(hovertemplate='%{y:.4f}')
fig.update_yaxes(title_text='ton', row=1, col=1)
fig.update_yaxes(title_text='ton', row=2, col=1)
fig.update_layout(height=600, hovermode='x unified')
fig.show()

## 6. Peramalan dengan Recurrent SSA (R-SSA)
Deret hasil rekonstruksi memenuhi **Linear Recurrent Relation (LRR)**:

$$y_t = \sum_{j=1}^{L-1} a_j\, y_{t-j}$$

Koefisiennya dihitung dari $r$ vektor eigen pertama. Misalkan $\pi_i$ adalah elemen terakhir $U_i$, $U_i^{\nabla}$ adalah $U_i$ tanpa elemen terakhir, dan $\nu^2 = \sum_{i=1}^{r} \pi_i^2$ adalah *verticality coefficient*. Maka

$$\mathcal{R} = (a_{L-1}, \dots, a_1)^T = \frac{1}{1 - \nu^2} \sum_{i=1}^{r} \pi_i U_i^{\nabla}$$

Ramalan dimulai dari ujung deret rekonstruksi. Setiap nilai ramalan dipakai lagi untuk meramal hari berikutnya. Syaratnya $\nu^2 < 1$.

In [25]:
R, nu2 = ssa.lrr_coefficients(model['U'], r)
print(f'Verticality coefficient ν² = {nu2:.6f} (< 1, R-SSA dapat dihitung)')
koef = pd.DataFrame({'Koefisien': R[::-1]}, index=pd.Index([f'a_{j} (y_t-{j})' for j in range(1, L)], name='Lag'))
koef

Verticality coefficient ν² = 0.206339 (< 1, R-SSA dapat dihitung)


,Koefisien
Lag,
a_1 (y_t-1),0.2480
a_2 (y_t-2),0.2241
a_3 (y_t-3),0.2039
a_4 (y_t-4),0.1748
a_5 (y_t-5),0.1527
a_6 (y_t-6),0.1245
a_7 (y_t-7),0.0924
a_8 (y_t-8),0.0600
a_9 (y_t-9),0.0319


In [26]:
jendela = model['signal'][-(L - 1):]
y_manual = float(np.dot(R, jendela))
print(f'Ramalan hari pertama data uji ({tgl_uji.iloc[0]:%d-%m-%Y}):')
print(f'  y = Σ a_j · y_(t-j) dari {L - 1} nilai rekonstruksi terakhir = {y_manual:.4f} ton')

pred = ssa.recurrent_forecast(model['signal'], R, n_uji)
print(f'  hasil fungsi recurrent_forecast                       = {pred[0]:.4f} ton')
print(f'  nilai aktual                                          = {uji[0]:.4f} ton')

Ramalan hari pertama data uji (20-10-2024):
  y = Σ a_j · y_(t-j) dari 15 nilai rekonstruksi terakhir = 63.3398 ton
  hasil fungsi recurrent_forecast                       = 63.3398 ton
  nilai aktual                                          = 64.0000 ton


## 7. Evaluasi pada Data Uji
Ramalan R-SSA untuk periode data uji dibandingkan dengan nilai aktualnya ($X_t$ = aktual, $Y_t$ = ramalan):

$$\text{MAE} = \frac{1}{n}\sum |X_t - Y_t| \qquad \text{MSE} = \frac{1}{n}\sum (X_t - Y_t)^2 \qquad \text{RMSE} = \sqrt{\text{MSE}} \qquad \text{MAPE} = \frac{100\%}{n}\sum \left|\frac{X_t - Y_t}{X_t}\right|$$

In [27]:
hasil_uji = pd.DataFrame({
    'tanggal': tgl_uji.dt.date.to_numpy(),
    'aktual (X)': uji,
    'ramalan (Y)': pred,
})
hasil_uji['error (X-Y)'] = hasil_uji['aktual (X)'] - hasil_uji['ramalan (Y)']
hasil_uji['|error|'] = hasil_uji['error (X-Y)'].abs()
hasil_uji['error²'] = hasil_uji['error (X-Y)'] ** 2
hasil_uji['APE (%)'] = hasil_uji['|error|'] / hasil_uji['aktual (X)'] * 100
hasil_uji.index = pd.RangeIndex(1, n_uji + 1, name='t')

with pd.option_context('display.max_rows', 200):
    display(hasil_uji)

,tanggal,aktual (X),ramalan (Y),error (X-Y),|error|,error²,APE (%)
t,,,,,,,
1,2024-10-20,64.0000,63.3398,0.6602,0.6602,0.4358,1.0315
2,2024-10-21,68.0000,63.1110,4.8890,4.8890,23.9019,7.1896
3,2024-10-22,66.0000,62.8941,3.1059,3.1059,9.6467,4.7059
4,2024-10-23,66.0000,62.6911,3.3089,3.3089,10.9485,5.0134
5,2024-10-24,64.0000,62.5083,1.4917,1.4917,2.2251,2.3308
6,2024-10-25,64.0000,62.3442,1.6558,1.6558,2.7415,2.5871
7,2024-10-26,61.0000,62.1912,-1.1912,1.1912,1.4190,1.9528
8,2024-10-27,61.0000,62.0528,-1.0528,1.0528,1.1085,1.7260
9,2024-10-28,61.0000,61.9165,-0.9165,0.9165,0.8399,1.5024


In [28]:
ev = ssa.evaluate(x, L, r, RASIO_LATIH)
assert np.allclose(ev['test_pred'], pred)

evaluasi = pd.DataFrame({
    'Perhitungan dari tabel': [hasil_uji['|error|'].mean(), hasil_uji['error²'].mean(),
                               np.sqrt(hasil_uji['error²'].mean()), hasil_uji['APE (%)'].mean()],
    'Fungsi ssa.metrics': [ev['mae'], ev['mse'], ev['rmse'], ev['mape']],
    'Data latih (rekonstruksi)': [ev['fit_metrics'][k] for k in ('mae', 'mse', 'rmse', 'mape')],
}, index=['MAE (ton)', 'MSE (ton²)', 'RMSE (ton)', 'MAPE (%)'])
display(evaluasi)

kriteria = pd.DataFrame({'Nilai MAPE': ['x < 10%', '10% ≤ x < 20%', '20% ≤ x < 50%', 'x ≥ 50%'],
                         'Kriteria': ['Sangat Baik', 'Baik', 'Layak', 'Buruk']})
print('Tabel 2.1 Kriteria nilai MAPE:')
display(kriteria)
print(f'MAPE data uji = {ev["mape"]:.4f}%  ->  kemampuan peramalan {ssa.mape_criteria(ev["mape"]).upper()}')

,Perhitungan dari tabel,Fungsi ssa.metrics,Data latih (rekonstruksi)
MAE (ton),2.3062,2.3062,1.9398
MSE (ton²),8.2150,8.2150,5.4368
RMSE (ton),2.8662,2.8662,2.3317
MAPE (%),3.7877,3.7877,3.1815


Tabel 2.1 Kriteria nilai MAPE:


,Nilai MAPE,Kriteria
0,x < 10%,Sangat Baik
1,10% ≤ x < 20%,Baik
2,20% ≤ x < 50%,Layak
3,x ≥ 50%,Buruk


MAPE data uji = 3.7877%  ->  kemampuan peramalan SANGAT BAIK


In [29]:
fig = go.Figure()
fig.add_scatter(x=tgl_latih, y=latih, name='Aktual (latih)', mode='lines', line=dict(color='lightgray'))
fig.add_scatter(x=tgl_latih, y=model['signal'], name='Rekonstruksi', mode='lines', line=dict(color='seagreen'))
fig.add_scatter(x=tgl_uji, y=uji, name='Aktual (uji)', mode='lines+markers', marker_size=4,
                line=dict(color='royalblue'))
fig.add_scatter(x=tgl_uji, y=pred, name='Ramalan R-SSA', mode='lines', line=dict(color='crimson', width=3))
fig.add_vline(x=tgl_uji.iloc[0], line_dash='dot', line_color='gray')
fig.update_traces(hovertemplate='%{y:.4f} ton')
fig.update_layout(title=f'Ramalan Data Uji (L={L}, r={r}, MAPE={ev["mape"]:.2f}%)', xaxis_title='Tanggal',
                  yaxis_title='Produksi (ton)', hovermode='x unified', height=480, xaxis_rangeslider_visible=True)
fig.show()

In [30]:
fig = px.bar(hasil_uji.reset_index(), x='tanggal', y='APE (%)', color='APE (%)', color_continuous_scale='OrRd',
             title='Absolute Percentage Error per Hari pada Data Uji')
fig.add_hline(y=ev['mape'], line_dash='dash', annotation_text=f'MAPE = {ev["mape"]:.2f}%')
fig.update_traces(hovertemplate='%{x|%d %b %Y}<br>APE = %{y:.4f}%<extra></extra>')
fig.update_layout(height=420)
fig.show()

## 8. Peramalan Periode Mendatang
Setelah akurasi diketahui, SSA dengan $L$ dan $r$ yang sama **dilatih ulang pada seluruh data** (latih + uji). Hasilnya dipakai untuk meramal 365 hari setelah data terakhir.

In [31]:
akhir = ssa.forecast(x, L, r, HORIZON)
tgl_ramal = pd.date_range(df.tanggal.iloc[-1] + pd.Timedelta(days=1), periods=HORIZON, freq='D')
ramalan = pd.DataFrame({'tanggal': tgl_ramal, 'ramalan (ton)': akhir['forecast']})

print(f'Model akhir: N = {N}, L = {L}, K = {akhir["K"]}, r = {r}, ν² = {akhir["nu2"]:.6f}')
print(f'Periode ramalan: {tgl_ramal[0]:%d-%m-%Y} s.d. {tgl_ramal[-1]:%d-%m-%Y} ({HORIZON} hari)')
print(f'Total ramalan  : {ramalan["ramalan (ton)"].sum():,.2f} ton')
print(f'Rata-rata/hari : {ramalan["ramalan (ton)"].mean():,.4f} ton')
display(ramalan.head(10))

Model akhir: N = 366, L = 16, K = 351, r = 2, ν² = 0.199461
Periode ramalan: 01-01-2025 s.d. 31-12-2025 (365 hari)
Total ramalan  : 20,628.28 ton
Rata-rata/hari : 56.5158 ton


,tanggal,ramalan (ton)
0,2025-01-01,58.8654
1,2025-01-02,58.8562
2,2025-01-03,58.8484
3,2025-01-04,58.8373
4,2025-01-05,58.8116
5,2025-01-06,58.7716
6,2025-01-07,58.7233
7,2025-01-08,58.6742
8,2025-01-09,58.6281
9,2025-01-10,58.5893


In [32]:
fig = go.Figure()
fig.add_scatter(x=df.tanggal, y=x, name='Aktual 2024', mode='lines', line=dict(color='lightgray'))
fig.add_scatter(x=df.tanggal, y=akhir['signal'], name='Rekonstruksi', mode='lines', line=dict(color='seagreen'))
fig.add_scatter(x=tgl_ramal, y=akhir['forecast'], name='Ramalan', mode='lines', line=dict(color='crimson', width=3))
fig.add_vline(x=df.tanggal.iloc[-1], line_dash='dot', line_color='gray')
fig.update_traces(hovertemplate='%{y:.4f} ton')
fig.update_layout(title=f'Ramalan Produksi Padi {HORIZON} Hari ke Depan', xaxis_title='Tanggal',
                  yaxis_title='Produksi (ton)', hovermode='x unified', height=480, xaxis_rangeslider_visible=True)
fig.show()

In [33]:
def rekap(data, freq):
    periode = data['tanggal'].dt.to_period(freq)
    hasil = data.groupby(periode)['ramalan (ton)'].agg(hari='count', total='sum', rata_rata='mean')
    hasil.index = [f'{NAMA_BULAN[p.month - 1]} {p.year}' if freq == 'M' else f'Kuartal {p.quarter} {p.year}'
                   for p in hasil.index]
    hasil.index.name = 'Periode'
    return hasil

rekap_bulan = rekap(ramalan, 'M')
rekap_kuartal = rekap(ramalan, 'Q')
print('Rekap ramalan per bulan:')
display(rekap_bulan)
print('Rekap ramalan per kuartal:')
display(rekap_kuartal)

Rekap ramalan per bulan:


,hari,total,rata_rata
Periode,,,
Januari 2025,31,"1,813.4606",58.4987
Februari 2025,28,"1,622.9813",57.9636
Maret 2025,31,"1,785.4653",57.5957
April 2025,30,"1,717.9123",57.2637
Mei 2025,31,"1,765.3658",56.9473
Juni 2025,30,"1,699.0871",56.6362
Juli 2025,31,"1,746.1655",56.3279
Agustus 2025,31,"1,736.5130",56.0165
September 2025,30,"1,671.3586",55.7120


Rekap ramalan per kuartal:


,hari,total,rata_rata
Periode,,,
Kuartal 1 2025,90,"5,221.9072",58.0212
Kuartal 2 2025,91,"5,182.3652",56.9491
Kuartal 3 2025,92,"5,154.0372",56.0221
Kuartal 4 2025,92,"5,069.9664",55.1083


In [34]:
fig = go.Figure()
fig.add_bar(x=bulanan.index, y=bulanan['total'], name='Aktual 2024',
            hovertemplate='%{x} 2024<br>%{y:,.2f} ton<extra></extra>')
fig.add_bar(x=[p.split()[0] for p in rekap_bulan.index], y=rekap_bulan['total'], name='Ramalan',
            customdata=rekap_bulan.index, hovertemplate='%{customdata}<br>%{y:,.2f} ton<extra></extra>')
fig.update_layout(barmode='group', title='Total Produksi per Bulan: Aktual 2024 vs Ramalan',
                  xaxis_title='Bulan', yaxis_title='Total produksi (ton)', height=450)
fig.show()

## 9. Ringkasan

In [35]:
ringkasan = pd.Series({
    'Jumlah data (N)': f'{N} hari',
    'Data latih / uji': f'{n_latih} / {n_uji} hari',
    'Window length (L)': L,
    'Grouping effect (r)': f'{r} (komponen 1 = tren, 2..{r} = musiman)' if r > 1 else '1 (tren)',
    'Kontribusi komponen terpilih': f'{svd_tabel["Kumulatif (%)"].iloc[r - 1]:.4f}%',
    'MAE data uji': f'{ev["mae"]:.4f} ton',
    'RMSE data uji': f'{ev["rmse"]:.4f} ton',
    'MAPE data uji': f'{ev["mape"]:.4f}%',
    'Kriteria (Tabel 2.1)': ssa.mape_criteria(ev['mape']),
    'Periode ramalan': f'{tgl_ramal[0]:%d-%m-%Y} s.d. {tgl_ramal[-1]:%d-%m-%Y}',
    'Total ramalan': f'{ramalan["ramalan (ton)"].sum():,.2f} ton',
}, name='Hasil')
ringkasan.to_frame()

,Hasil
Jumlah data (N),366 hari
Data latih / uji,293 / 73 hari
Window length (L),16
Grouping effect (r),"2 (komponen 1 = tren, 2..2 = musiman)"
Kontribusi komponen terpilih,99.8446%
MAE data uji,2.3062 ton
RMSE data uji,2.8662 ton
MAPE data uji,3.7877%
Kriteria (Tabel 2.1),Sangat Baik
Periode ramalan,01-01-2025 s.d. 31-12-2025
